# OBEDIT-4D single-runtime Colab workflow

This notebook uses one modern PyTorch runtime for 4DGS rendering, the object editor, Gaussian fitting, and final rendering. Run cells in order. You must provide a trained 4DGS checkpoint and synchronized rendered PNGs/masks before the editor cell.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
from pathlib import Path
PROJECT_ROOT = Path('/content/drive/MyDrive/OBEDIT-4D')
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
print(PROJECT_ROOT)

## Cell 3: select a GPU

Use Runtime > Change runtime type > T4 GPU or a better CUDA GPU. The next cell must report CUDA available.

In [ ]:
import torch
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Select a CUDA GPU runtime before continuing')
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

In [ ]:
%cd /content
!rm -rf /content/OBEDIT-4D
!git clone --branch object_editor https://github.com/baibhavsingh021/OBEDIT-4D.git OBEDIT-4D
%cd /content/OBEDIT-4D
!git submodule update --init --recursive
!git branch --show-current

## Cell 5: install the unified environment

Do not install the root `requirements.txt`; its old Diffusers pins conflict with OmniGen.

In [ ]:
%cd /content/OBEDIT-4D
!apt-get update -qq
!apt-get install -y -qq libglm-dev
!pip install -q -r object_editor/requirements-colab.txt
!pip install -q --no-deps 'git+https://github.com/VectorSpaceLab/OmniGen.git'
!pip install -q -e submodules/depth-diff-gaussian-rasterization
!pip install -q -e submodules/simple-knn

In [ ]:
# Restart the runtime manually if the previous cell replaced Torch.
import torch
assert torch.cuda.is_available(), 'CUDA is unavailable after installation'
print('Unified runtime ready:', torch.__version__, torch.version.cuda)

## Cell 8: configure persistent paths

Set these paths to your trained scene and input folders. `images_dir` and `mask_dir` must contain the same number of PNGs in matching sorted order.

In [ ]:
from pathlib import Path
SCENE = 'cook_spinach'
DATASET = 'dynerf'
IMAGES_DIR = PROJECT_ROOT / 'inputs' / 'images'
MASK_DIR = PROJECT_ROOT / 'inputs' / 'masks'
EDITED_ROOT = PROJECT_ROOT / 'edited'
MODEL_PATH = PROJECT_ROOT / 'output' / DATASET / SCENE
for path in (IMAGES_DIR, MASK_DIR, MODEL_PATH):
    print(path, path.exists())
image_paths = sorted(IMAGES_DIR.glob('*.png'))
mask_paths = sorted(MASK_DIR.glob('*.png'))
print('images:', len(image_paths), 'masks:', len(mask_paths))
if len(image_paths) == 0 or len(image_paths) != len(mask_paths):
    raise ValueError('Provide equal nonzero PNG image and mask folders before editing')

## Cell 9: prerequisite render/mask handoff

If the input folders are empty, use your existing trained-4DGS renderer and Grounded-SAM/manual-mask workflow first. The current branch does not silently invent masks or render a scene from only a checkpoint. A mask is grayscale, target pixels are nonzero, and its resolution matches its RGB image.

In [ ]:
%cd /content/OBEDIT-4D
!python -m object_editor.scripts.edit_object \
  --images_dir $IMAGES_DIR \
  --mask_dir $MASK_DIR \
  --output_dir $EDITED_ROOT \
  --editor_model omnigen \
  --editor_ckpt Shitao/OmniGen-v1 \
  --target_query 'the red water bottle' \
  --edit_instruction 'make the water bottle metallic blue' \
  --edit_type appearance \
  --preservation_mode strict \
  --coupling_strength 0.7

In [ ]:
# Set this to the exact hash printed by the editor cell.
RUN_NAME = 'edit_appearance_REPLACE_HASH'
EDITED_DIR = EDITED_ROOT / RUN_NAME
print('Edited directory:', EDITED_DIR)
print('Edited PNG count:', len(list(EDITED_DIR.glob('*.png'))))

## Cell 12: fit the edited canonical Gaussians

Replace the PLY path and scene config with your trained checkpoint. This continues in the same runtime and passes the editor output directly through `--edited_images_path`.

In [ ]:
%cd /content/OBEDIT-4D
!python edit_3d.py \
  --configs arguments/dynerf/cook_spinach.py \
  --dataset dynerf \
  --scene cook_spinach \
  --prompt 'make the target metallic blue' \
  --target_query 'the red water bottle' \
  --edit_instruction 'make the target metallic blue' \
  --edit_type appearance \
  --run_name $RUN_NAME \
  --edited_images_path $EDITED_DIR \
  --ply_path /content/drive/MyDrive/OBEDIT-4D/output/dynerf/cook_spinach/point_cloud/REPLACE/point_cloud.ply

## Cell 13: render the final edited 4DGS

The output directory is controlled by the existing renderer/checkpoint conventions. Inspect multiple views and timesteps, including protected regions and disocclusions.

In [ ]:
%cd /content/OBEDIT-4D
!python render_edited4d.py \
  --configs arguments/dynerf/cook_spinach.py \
  --ply_path /content/drive/MyDrive/OBEDIT-4D/output/dynerf/cook_spinach/point_cloud_refine/$RUN_NAME/iteration_800/point_cloud.ply \
  -s /content/drive/MyDrive/OBEDIT-4D/data/dynerf/cook_spinach \
  --model_path /content/drive/MyDrive/OBEDIT-4D/output/dynerf/cook_spinach